# Fine-Tuning Math — Theory Notebook

Interactive implementations covering:

1. **Full Fine-Tuning** — discriminative LR, gradual unfreezing, overfitting dynamics
2. **LoRA** — low-rank adaptation, parameter counting, forward pass, merging
3. **QLoRA** — NF4 quantisation, memory calculation
4. **DoRA** — weight decomposition (magnitude + direction)
5. **DPO** — direct preference optimisation loss and gradient
6. **GRPO** — group relative policy optimisation
7. **Instruction Tuning** — loss masking, IFD score
8. **EWC** — elastic weight consolidation
9. **Model Merging** — weight averaging, task vectors, SLERP
10. **Summary**

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_vector(name, v, decimals=6):
    formatted = ', '.join(f'{x:.{decimals}f}' for x in v)
    print(f"  {name} = [{formatted}]")

def print_table(headers, rows):
    widths = [max(len(str(h)), max(len(str(r[i])) for r in rows)) + 2 for i, h in enumerate(headers)]
    header_str = ''.join(f'{h:<{w}}' for h, w in zip(headers, widths))
    print(header_str)
    print('-' * sum(widths))
    for row in rows:
        print(''.join(f'{str(v):<{w}}' for v, w in zip(row, widths)))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

print("Setup complete ✓")

## §1 Full Fine-Tuning — Discriminative LR & Overfitting

**Discriminative fine-tuning** (ULMFiT): different learning rates per layer:
$$\eta^l = \frac{\eta}{\gamma^{N-l}}, \quad l = 1, \ldots, N$$

**Gradual unfreezing**: start frozen; unfreeze layers progressively.

Key risk: fine-tuning datasets are tiny relative to model capacity → **overfitting**.

In [ ]:
# === §1: Full Fine-Tuning ===

# --- 1a: Discriminative Learning Rates ---
print("=== Discriminative Fine-Tuning (ULMFiT) ===\n")
print("η^l = η / γ^(N-l)  where γ = 2.6\n")

eta_base = 1e-4
gamma = 2.6
N = 12  # 12-layer model

print(f"{'Layer':<8} {'LR':<18} {'Ratio to top':<15} {'Role'}")
print("-" * 60)
for l in range(1, N+1):
    lr = eta_base / gamma**(N - l)
    ratio = lr / eta_base
    role = "embeddings" if l <= 2 else ("low-level" if l <= 6 else ("mid-level" if l <= 9 else "task-specific"))
    print(f"{l:<8} {lr:<18.2e} {ratio:<15.4f} {role}")

print(f"\nSpread: layer 1 LR is {gamma**(N-1):.0f}× smaller than layer {N}")
print("→ Lower layers (syntax) barely change; upper layers (task) adapt freely")

# --- 1b: Gradual Unfreezing Simulation ---
print("\n\n=== Gradual Unfreezing ===\n")
T_unfreeze = 500  # steps between unfreezing each layer

print(f"{'Step':<8} {'Layers Trainable':<25} {'Trainable %'}")
print("-" * 50)
for step in [0, 500, 1000, 2000, 3000, 5000, 6000]:
    unfrozen = min(N, max(1, 1 + step // T_unfreeze))
    layers = f"{N-unfrozen+1}–{N}" if unfrozen > 1 else f"{N} only"
    pct = unfrozen / N * 100
    print(f"{step:<8} {layers:<25} {pct:.0f}%")

# --- 1c: Overfitting Simulation ---
print("\n\n=== Overfitting in Fine-Tuning ===\n")
np.random.seed(42)

# Simulate: true function is quadratic; model is high-capacity polynomial
n_train = 50  # tiny fine-tuning dataset
x_train = np.sort(np.random.uniform(-1, 1, n_train))
y_true = 0.5 * x_train**2 - 0.3 * x_train + 0.1
y_train = y_true + np.random.randn(n_train) * 0.1  # noisy labels

x_val = np.linspace(-1, 1, 200)
y_val_true = 0.5 * x_val**2 - 0.3 * x_val + 0.1

# "Fine-tune" with increasing complexity (epochs ~ polynomial degree)
print(f"{'Degree':<10} {'Train MSE':<14} {'Val MSE':<14} {'Status'}")
print("-" * 50)
train_mses, val_mses = [], []
for deg in [1, 2, 3, 5, 10, 20, 40]:
    coeffs = np.polyfit(x_train, y_train, deg)
    y_pred_train = np.polyval(coeffs, x_train)
    y_pred_val = np.polyval(coeffs, x_val)
    train_mse = np.mean((y_pred_train - y_train)**2)
    val_mse = np.mean((y_pred_val - y_val_true)**2)
    train_mses.append(train_mse)
    val_mses.append(val_mse)
    status = "✓ good" if val_mse < 0.01 else ("⚠ overfit" if train_mse < val_mse * 0.1 else "↗ improving")
    print(f"{deg:<10} {train_mse:<14.6f} {val_mse:<14.6f} {status}")

# --- 1d: Memory cost table ---
print("\n\n=== Full Fine-Tuning Memory Cost (16N bytes) ===\n")
models = [
    ("LLaMA-3 8B", 8e9),
    ("LLaMA-3 70B", 70e9),
    ("LLaMA-3 405B", 405e9),
]
print(f"{'Model':<18} {'Params':<10} {'Memory (GB)':<14} {'A100-80GB GPUs'}")
print("-" * 56)
for name, N in models:
    mem_gb = 16 * N / 1e9
    gpus = int(np.ceil(mem_gb / 70))  # ~70GB usable
    print(f"{name:<18} {N/1e9:.0f}B{'':<5} {mem_gb:<14.0f} {gpus}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    degs = [1, 2, 3, 5, 10, 20, 40]
    ax.plot(degs, train_mses, 'b-o', label='Train MSE', linewidth=2)
    ax.plot(degs, val_mses, 'r-s', label='Val MSE', linewidth=2)
    ax.axhline(0.01, color='green', linestyle='--', alpha=0.5, label='Good threshold')
    ax.set_xlabel('Model Complexity (polynomial degree)')
    ax.set_ylabel('MSE'); ax.set_yscale('log')
    ax.set_title('Overfitting in Fine-Tuning (Small Dataset)')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('overfitting_finetune.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: overfitting_finetune.png")

## §2 LoRA — Low-Rank Adaptation

**Core idea:** Instead of updating full $W \in \mathbb{R}^{d \times d}$, learn low-rank update:

$$W' = W_0 + \Delta W = W_0 + BA, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times d}, \; r \ll d$$

**Key properties:**
- Initialization: $A \sim \mathcal{N}(0, \sigma^2)$, $B = 0$ → $\Delta W = 0$ at start
- Scaling: output multiplied by $\alpha / r$ (typically $\alpha = 2r$)
- Parameter count per matrix: $r(d_{\text{in}} + d_{\text{out}})$ vs $d_{\text{in}} \times d_{\text{out}}$
- Merging: at inference, $W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA$ — **zero added latency**

In [ ]:
# === §2: LoRA Implementation ===

# --- 2a: LoRA Forward Pass ---
print("=== LoRA Forward Pass ===\n")
np.random.seed(42)

d = 64     # hidden dim (small for demo)
r = 4      # rank
alpha = 8  # scaling factor
scale = alpha / r

# Pretrained weight (frozen)
W0 = np.random.randn(d, d) * 0.02

# LoRA matrices: A ~ N(0, σ²), B = 0
A = np.random.randn(r, d) * 0.01  # (r x d)
B = np.zeros((d, r))               # (d x r) → BA = 0 at init

print(f"W0 shape: {W0.shape}  (frozen, {d*d:,} params)")
print(f"A shape:  {A.shape}   ({r*d:,} params)")
print(f"B shape:  {B.shape}   ({d*r:,} params)")
print(f"LoRA params: {2*r*d:,} ({2*r*d/(d*d)*100:.1f}% of full)")
print(f"Scale α/r = {alpha}/{r} = {scale}")

# Forward pass: h = W0·x + (α/r)·B·A·x
x = np.random.randn(d)
h_frozen = W0 @ x            # pretrained output
h_lora   = W0 @ x + scale * (B @ (A @ x))  # LoRA output

print(f"\n‖W0·x‖ = {np.linalg.norm(h_frozen):.4f}")
print(f"‖LoRA output‖ = {np.linalg.norm(h_lora):.4f}")
print(f"Δ output at init: {np.linalg.norm(h_lora - h_frozen):.6f}  (≈ 0 because B=0)")

# --- 2b: Simulate Training (gradient step) ---
print("\n\n=== LoRA Training Simulation ===\n")

# Target: fine-tuned matrix (what full FT would give)
W_target = W0 + np.random.randn(d, d) * 0.005

# "Train" via gradient descent on ||W0 + BA - W_target||²
lr = 0.01
losses = []
for step in range(200):
    # Forward: residual = (W0 + scale * B@A) - W_target
    delta_W = scale * B @ A
    residual = W0 + delta_W - W_target
    loss = np.sum(residual**2) / (d*d)
    losses.append(loss)
    
    # Gradients w.r.t. A and B
    grad_delta = 2 * residual / (d*d)  # (d x d)
    grad_B = scale * grad_delta @ A.T  # (d x r)
    grad_A = scale * B.T @ grad_delta  # (r x d)
    
    B -= lr * grad_B
    A -= lr * grad_A

print(f"{'Step':<8} {'Loss':<14} {'ΔW Frobenius'}")
print("-" * 40)
for s in [0, 10, 50, 100, 199]:
    delta_W = scale * B @ A
    err = np.linalg.norm(delta_W - (W_target - W0)) / np.linalg.norm(W_target - W0)
    print(f"{s:<8} {losses[s]:<14.6f} {err:.4f}" if s < 199 else f"{s:<8} {losses[s]:<14.6f} {err:.4f}")

# --- 2c: Rank analysis via SVD ---
print("\n\n=== Low-Rank Structure of ΔW ===\n")
delta_actual = W_target - W0
U, S, Vt = np.linalg.svd(delta_actual)

print(f"{'Rank k':<10} {'Explained var %':<18} {'Reconstruction err'}")
print("-" * 50)
total_energy = np.sum(S**2)
for k in [1, 2, 4, 8, 16, 32, d]:
    explained = np.sum(S[:k]**2) / total_energy * 100
    reconst = np.sqrt(np.sum(S[k:]**2)) / np.sqrt(total_energy)
    print(f"{k:<10} {explained:<18.2f} {reconst:.4f}")

print(f"\n→ rank {r} captures {np.sum(S[:r]**2)/total_energy*100:.1f}% of ΔW variance")
print(f"→ This is why LoRA works: fine-tuning updates are low intrinsic rank!")

# --- 2d: Parameter Count Calculator ---
print("\n\n=== LoRA Parameter Count: Real Models ===\n")
configs = [
    ("LLaMA-3 8B", 4096, 32, "Q,V", 2),
    ("LLaMA-3 8B", 4096, 32, "Q,K,V,O", 4),
    ("LLaMA-3 70B", 8192, 80, "Q,V", 2),
    ("LLaMA-3 70B", 8192, 80, "all attn+mlp", 7),
]
print(f"{'Model':<16} {'d':<6} {'L':<4} {'Matrices':<16} {'r':<4} {'LoRA params':<14} {'% of total'}")
print("-" * 75)
for name, d_model, n_layers, matrices, n_mats in configs:
    for rank in [8, 16, 64]:
        lora_params = n_mats * n_layers * 2 * rank * d_model
        total = 8e9 if "8B" in name else 70e9
        pct = lora_params / total * 100
        print(f"{name:<16} {d_model:<6} {n_layers:<4} {matrices:<16} {rank:<4} {lora_params/1e6:<14.1f}M {pct:.3f}%")
    print()

# --- 2e: Merging Demo ---
print("=== LoRA Merging (Zero Inference Cost) ===\n")
W_merged = W0 + scale * B @ A
print(f"‖W0‖         = {np.linalg.norm(W0):.4f}")
print(f"‖scale·B·A‖  = {np.linalg.norm(scale * B @ A):.4f}")
print(f"‖W_merged‖   = {np.linalg.norm(W_merged):.4f}")
print(f"\nAt inference: just use W_merged — same cost as original model!")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    # Loss curve
    axes[0].plot(losses, 'b-', linewidth=1.5)
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('LoRA Training Loss'); axes[0].set_yscale('log')
    axes[0].grid(True, alpha=0.3)
    # Singular values
    axes[1].bar(range(min(20, len(S))), S[:20], color='steelblue')
    axes[1].axvline(r-0.5, color='red', linestyle='--', label=f'rank r={r}')
    axes[1].set_xlabel('Singular Value Index'); axes[1].set_ylabel('σᵢ')
    axes[1].set_title('Singular Values of ΔW'); axes[1].legend()
    # Heatmaps
    im = axes[2].imshow(scale * B @ A, cmap='RdBu', aspect='auto', 
                         vmin=-0.02, vmax=0.02)
    axes[2].set_title(f'Learned ΔW (rank {r})'); plt.colorbar(im, ax=axes[2])
    plt.tight_layout()
    plt.savefig('lora_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: lora_analysis.png")

## §3 QLoRA — Quantised LoRA

**Key innovations:**
1. **NF4 quantisation:** 4-bit format optimised for normally-distributed weights
2. **Double quantisation:** quantise the quantisation constants themselves (saves ~0.4 GB/65B)
3. **Paged optimisers:** unified memory to avoid OOM on long sequences

$$\text{Memory} \approx \frac{4N}{10^9} + \frac{2 \cdot r(d_{in}+d_{out}) \cdot L \cdot n_{mat} \cdot 2}{10^9} \;\text{GB}$$

where $4N/10^9$ = 4-bit base model, second term = FP16 LoRA adapters + optimizer states

In [ ]:
# === §3: QLoRA Implementation ===
np.random.seed(42)

# --- 3a: NF4 Quantisation Simulation ---
print("=== NF4 Quantisation ===\n")

# NF4 levels: optimal for N(0,1) distribution (16 levels)
nf4_levels = np.array([
    -1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848, -0.0911, 0.0,
    0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7230, 1.0
])

# Simulate quantising a weight matrix
d = 64
W = np.random.randn(d, d) * 0.02  # typical LLM weight scale
absmax = np.max(np.abs(W))
W_normalized = W / absmax  # scale to [-1, 1]

# Quantise: find nearest NF4 level
W_flat = W_normalized.flatten()
W_quant_idx = np.array([np.argmin(np.abs(nf4_levels - w)) for w in W_flat])
W_quant = nf4_levels[W_quant_idx].reshape(d, d) * absmax  # dequantise

# Error analysis
quant_error = np.abs(W - W_quant)
print(f"Original:     FP32, {d*d*4/1024:.1f} KB")
print(f"NF4:          4-bit, {d*d*0.5/1024:.1f} KB  ({d*d*0.5/(d*d*4)*100:.1f}% of FP32)")
print(f"Mean abs error: {np.mean(quant_error):.6f}")
print(f"Max abs error:  {np.max(quant_error):.6f}")
print(f"Relative error: {np.linalg.norm(W - W_quant)/np.linalg.norm(W)*100:.2f}%")

# Compare NF4 vs uniform quantisation
uniform_levels = np.linspace(-1, 1, 16)
W_uniform_idx = np.array([np.argmin(np.abs(uniform_levels - w)) for w in W_flat])
W_uniform = uniform_levels[W_uniform_idx].reshape(d, d) * absmax

print(f"\nNF4 vs Uniform 4-bit comparison:")
print(f"  NF4     MSE: {np.mean((W - W_quant)**2):.8f}")
print(f"  Uniform MSE: {np.mean((W - W_uniform)**2):.8f}")
print(f"  NF4 is {np.mean((W - W_uniform)**2)/np.mean((W - W_quant)**2):.1f}× better for normal weights")

# --- 3b: Double Quantisation ---
print("\n\n=== Double Quantisation ===\n")
block_size = 64  # quantise in blocks
n_blocks = (d * d) // block_size

# First quantisation: compute absmax per block
W_flat_full = W.flatten()
absmax_per_block = np.array([
    np.max(np.abs(W_flat_full[i*block_size:(i+1)*block_size]))
    for i in range(n_blocks)
])

# Second quantisation: quantise the absmax values themselves
absmax_absmax = np.max(absmax_per_block)
absmax_normalized = absmax_per_block / absmax_absmax

# 8-bit quantisation of the constants
int8_levels = np.linspace(0, 1, 256)
absmax_quant = np.array([int8_levels[np.argmin(np.abs(int8_levels - a))] for a in absmax_normalized])
absmax_dequant = absmax_quant * absmax_absmax

print(f"Blocks: {n_blocks} of size {block_size}")
print(f"Without double quant: {n_blocks * 4:.0f} bytes for constants (FP32)")
print(f"With double quant:    {n_blocks * 1 + 4:.0f} bytes for constants (INT8 + 1 FP32)")
print(f"Savings per block:    {(4 - 1)/4*100:.0f}%")

# --- 3c: Memory Calculator ---
print("\n\n=== QLoRA Memory Calculator ===\n")
models = [
    ("LLaMA-3 8B", 8e9, 4096, 32, 32),
    ("LLaMA-3 70B", 70e9, 8192, 80, 64),
    ("Mixtral 8x7B", 46.7e9, 4096, 32, 32),
    ("LLaMA-3 405B", 405e9, 16384, 126, 128),
]

print(f"{'Model':<18} {'Base(4bit)':<12} {'LoRA(FP16)':<12} {'Optim':<10} {'Total GB':<10} {'GPUs(80G)'}")
print("-" * 72)
for name, N, d_model, n_layers, r in models:
    base_gb = 4 * N / (8 * 1e9)     # 4-bit = 0.5 bytes per param
    n_mats = 4  # Q,K,V,O
    lora_params = n_mats * n_layers * 2 * r * d_model
    lora_gb = lora_params * 2 / 1e9   # FP16
    optim_gb = lora_params * 8 / 1e9  # Adam: 2×FP32 moments
    total = base_gb + lora_gb + optim_gb
    gpus = int(np.ceil(total / 70))
    print(f"{name:<18} {base_gb:<12.1f} {lora_gb:<12.2f} {optim_gb:<10.2f} {total:<10.1f} {gpus}")

# Comparison with full fine-tuning
print(f"\n{'Method':<20} {'8B Memory':<14} {'70B Memory':<14} {'Savings'}")
print("-" * 55)
for method, factor in [("Full FT (FP32)", 16), ("Full FT (BF16)", 12),
                        ("LoRA (FP16 base)", 2+0.1), ("QLoRA (NF4)", 0.5+0.1)]:
    mem_8b = factor * 8
    mem_70b = factor * 70
    savings = (1 - factor/16) * 100
    print(f"{method:<20} {mem_8b:<14.1f} {mem_70b:<14.1f} {savings:.0f}%")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # NF4 vs Uniform
    x_pts = np.linspace(-1, 1, 1000)
    axes[0].hist(W_normalized.flatten(), bins=50, density=True, alpha=0.5, label='Weight dist')
    for lv in nf4_levels:
        axes[0].axvline(lv, color='red', alpha=0.3, linewidth=0.8)
    for lv in uniform_levels:
        axes[0].axvline(lv, color='blue', alpha=0.15, linewidth=0.8)
    axes[0].set_title('NF4 (red) vs Uniform (blue) Quantisation Levels')
    axes[0].set_xlabel('Normalised weight value'); axes[0].legend()
    # Memory comparison
    methods = ['Full FT\n(FP32)', 'Full FT\n(BF16)', 'LoRA\n(FP16)', 'QLoRA\n(NF4)']
    mem_8b = [128, 96, 16.8, 4.8]
    mem_70b = [1120, 840, 147, 42]
    x_pos = np.arange(len(methods))
    axes[1].bar(x_pos - 0.15, mem_8b, 0.3, label='8B model', color='steelblue')
    axes[1].bar(x_pos + 0.15, np.log10(mem_70b), 0.3, label='70B (log₁₀)', color='coral', alpha=0.7)
    axes[1].set_xticks(x_pos); axes[1].set_xticklabels(methods)
    axes[1].set_ylabel('Memory (GB)'); axes[1].set_title('Fine-Tuning Memory Requirements')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig('qlora_memory.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: qlora_memory.png")

## §4 DoRA — Weight-Decomposed Low-Rank Adaptation

**Key insight:** Decompose weight into **magnitude** and **direction**, then apply LoRA only to direction:

$$W' = m \cdot \frac{W_0 + BA}{\|W_0 + BA\|_c}$$

where $m \in \mathbb{R}^{d_{out}}$ is a learnable magnitude vector and $\|\cdot\|_c$ is column-wise norm.

This mimics how full fine-tuning works (changes magnitude and direction independently), while LoRA couples them.

In [ ]:
# === §4: DoRA Implementation ===
np.random.seed(42)

print("=== DoRA: Weight-Decomposed Low-Rank Adaptation ===\n")
d = 32
r = 4

# Pretrained weight
W0 = np.random.randn(d, d) * 0.02

# --- Decompose W0 into magnitude and direction ---
col_norms = np.linalg.norm(W0, axis=0, keepdims=True)  # (1, d)
m_init = col_norms.flatten()       # magnitude vector (d,)
V_init = W0 / col_norms            # unit direction matrix (d, d)

print(f"W0 decomposition: W0 = m ⊙ V/‖V‖_c")
print(f"  m (magnitude): shape {m_init.shape}, mean={np.mean(m_init):.4f}")
print(f"  V (direction):  shape {V_init.shape}, columns are unit vectors")
print(f"  Verify: ‖W0 - m·V‖ = {np.linalg.norm(W0 - m_init * V_init):.2e}")

# --- LoRA vs DoRA comparison ---
print("\n=== LoRA vs DoRA: Learning Dynamics ===\n")

# Target weight (what we want after fine-tuning)
W_target = W0 + np.random.randn(d, d) * 0.01

# LoRA training
A_lora = np.random.randn(r, d) * 0.01
B_lora = np.zeros((d, r))
alpha, scale = 8, 8/r

# DoRA training
A_dora = np.random.randn(r, d) * 0.01
B_dora = np.zeros((d, r))
m_dora = m_init.copy()

lr = 0.01
lora_losses, dora_losses = [], []
lora_mag_changes, lora_dir_changes = [], []
dora_mag_changes, dora_dir_changes = [], []

for step in range(300):
    # --- LoRA forward ---
    W_lora = W0 + scale * B_lora @ A_lora
    loss_lora = np.sum((W_lora - W_target)**2) / (d*d)
    lora_losses.append(loss_lora)
    
    # Track magnitude/direction changes for LoRA
    lora_col_norms = np.linalg.norm(W_lora, axis=0)
    lora_mag_change = np.mean(np.abs(lora_col_norms - m_init))
    lora_dir_change = np.mean(1 - np.sum((W_lora / np.linalg.norm(W_lora, axis=0, keepdims=True)) * V_init, axis=0))
    lora_mag_changes.append(lora_mag_change)
    lora_dir_changes.append(lora_dir_change)
    
    # LoRA gradients
    grad = 2 * (W_lora - W_target) / (d*d)
    grad_B_lora = scale * grad @ A_lora.T
    grad_A_lora = scale * B_lora.T @ grad
    B_lora -= lr * grad_B_lora
    A_lora -= lr * grad_A_lora
    
    # --- DoRA forward ---
    V_dora = W0 + scale * B_dora @ A_dora
    V_dora_norm = V_dora / np.linalg.norm(V_dora, axis=0, keepdims=True)
    W_dora = m_dora[np.newaxis, :] * V_dora_norm
    loss_dora = np.sum((W_dora - W_target)**2) / (d*d)
    dora_losses.append(loss_dora)
    
    # Track DoRA changes
    dora_mag_change = np.mean(np.abs(m_dora - m_init))
    dora_dir_change = np.mean(1 - np.sum(V_dora_norm * V_init, axis=0))
    dora_mag_changes.append(dora_mag_change)
    dora_dir_changes.append(dora_dir_change)
    
    # DoRA gradients (simplified)
    grad_dora = 2 * (W_dora - W_target) / (d*d)
    grad_m = np.sum(grad_dora * V_dora_norm, axis=0)
    m_dora -= lr * grad_m
    grad_V = m_dora[np.newaxis, :] * grad_dora
    grad_B_dora = scale * grad_V @ A_dora.T
    grad_A_dora = scale * B_dora.T @ grad_V
    B_dora -= lr * grad_B_dora
    A_dora -= lr * grad_A_dora

print(f"{'Step':<8} {'LoRA Loss':<14} {'DoRA Loss':<14} {'Winner'}")
print("-" * 45)
for s in [0, 50, 100, 200, 299]:
    winner = "DoRA" if dora_losses[s] < lora_losses[s] else "LoRA"
    print(f"{s:<8} {lora_losses[s]:<14.6f} {dora_losses[s]:<14.6f} {winner}")

print(f"\nFinal magnitude change — LoRA: {lora_mag_changes[-1]:.5f}, DoRA: {dora_mag_changes[-1]:.5f}")
print(f"Final direction change — LoRA: {lora_dir_changes[-1]:.5f}, DoRA: {dora_dir_changes[-1]:.5f}")
print("→ DoRA decouples magnitude & direction, matching full FT behavior")

# --- DoRA overhead ---
print(f"\n=== DoRA Overhead ===")
print(f"Extra params vs LoRA: {d} (magnitude vector per matrix)")
print(f"LoRA params:  {2*r*d}")
print(f"DoRA params:  {2*r*d + d} ({(2*r*d+d)/(2*r*d)*100:.1f}% of LoRA)")
print(f"Overhead:     {d/(2*r*d)*100:.1f}%  — negligible!")

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(lora_losses, 'b-', label='LoRA', alpha=0.8)
    axes[0].plot(dora_losses, 'r-', label='DoRA', alpha=0.8)
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss'); axes[0].set_yscale('log')
    axes[0].set_title('LoRA vs DoRA: Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(lora_mag_changes, 'b-', label='LoRA mag Δ', alpha=0.7)
    axes[1].plot(dora_mag_changes, 'r-', label='DoRA mag Δ', alpha=0.7)
    axes[1].set_title('Magnitude Changes'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].plot(lora_dir_changes, 'b-', label='LoRA dir Δ', alpha=0.7)
    axes[2].plot(dora_dir_changes, 'r-', label='DoRA dir Δ', alpha=0.7)
    axes[2].set_title('Direction Changes'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('dora_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: dora_analysis.png")

## §5 DPO — Direct Preference Optimisation

**Key insight:** Eliminate the reward model entirely. The optimal policy under RLHF has a closed-form:

$$r^*(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

**DPO Loss** (substituting into Bradley-Terry):

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\!\left(\beta \left[\log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)\right]$$

**Gradient weighting:** Harder examples (where model is wrong) get larger gradients:
$$\nabla \mathcal{L} \propto \sigma(\hat{r}_l - \hat{r}_w) \cdot [\nabla\log\pi_\theta(y_w) - \nabla\log\pi_\theta(y_l)]$$

In [ ]:
# === §5: DPO Implementation ===
np.random.seed(42)

# --- 5a: DPO Loss Computation ---
print("=== DPO Loss: Step by Step ===\n")

def dpo_loss(log_pi_w, log_pi_l, log_ref_w, log_ref_l, beta):
    """Compute DPO loss for a single preference pair."""
    log_ratio_w = log_pi_w - log_ref_w  # log π(yw|x) / πref(yw|x)
    log_ratio_l = log_pi_l - log_ref_l  # log π(yl|x) / πref(yl|x)
    logit = beta * (log_ratio_w - log_ratio_l)
    loss = -np.log(sigmoid(logit))
    return loss, logit, log_ratio_w, log_ratio_l

# Example preference pair
log_pi_w  = -2.3   # model's log-prob for preferred response
log_pi_l  = -3.1   # model's log-prob for rejected response
log_ref_w = -2.5   # reference model's log-prob for preferred
log_ref_l = -2.8   # reference model's log-prob for rejected
beta = 0.1

loss, logit, ratio_w, ratio_l = dpo_loss(log_pi_w, log_pi_l, log_ref_w, log_ref_l, beta)

print(f"Preferred:  log π(yw|x) = {log_pi_w},  log πref(yw|x) = {log_ref_w}")
print(f"Rejected:   log π(yl|x) = {log_pi_l},  log πref(yl|x) = {log_ref_l}")
print(f"β = {beta}\n")
print(f"Step 1: log-ratio (win)  = {log_pi_w} - {log_ref_w} = {ratio_w:.2f}")
print(f"Step 2: log-ratio (lose) = {log_pi_l} - {log_ref_l} = {ratio_l:.2f}")
print(f"Step 3: β·(ratio_w - ratio_l) = {beta}·({ratio_w:.2f} - {ratio_l:.2f}) = {logit:.4f}")
print(f"Step 4: loss = -log σ({logit:.4f}) = {loss:.4f}")

# --- 5b: Gradient Weight Analysis ---
print("\n\n=== DPO Gradient Weighting ===\n")
print("Weight = σ(r̂_l - r̂_w): higher when model is WRONG\n")

# Simulate a batch of preference pairs
n_pairs = 8
log_pi_ws  = np.random.uniform(-4, -1, n_pairs)
log_pi_ls  = log_pi_ws + np.random.uniform(-2, 1, n_pairs)  # sometimes model prefers wrong one
log_ref_ws = log_pi_ws + np.random.uniform(-0.5, 0.5, n_pairs)
log_ref_ls = log_pi_ls + np.random.uniform(-0.5, 0.5, n_pairs)

print(f"{'Pair':<6} {'r̂_w':<10} {'r̂_l':<10} {'Grad weight':<14} {'Model correct?'}")
print("-" * 55)
for i in range(n_pairs):
    r_w = beta * (log_pi_ws[i] - log_ref_ws[i])
    r_l = beta * (log_pi_ls[i] - log_ref_ls[i])
    grad_weight = sigmoid(r_l - r_w)  # higher when model ranks wrong
    correct = "✓" if r_w > r_l else "✗ (needs more gradient)"
    print(f"{i+1:<6} {r_w:<10.4f} {r_l:<10.4f} {grad_weight:<14.4f} {correct}")

print("\n→ DPO automatically upweights examples the model gets wrong!")

# --- 5c: β Sweep ---
print("\n\n=== Effect of β (Temperature) ===\n")
betas = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]

# Fixed preference pair
lw, ll, rw, rl = -2.3, -3.0, -2.5, -2.8

print(f"{'β':<8} {'Loss':<10} {'σ(logit)':<12} {'KL penalty':<14} {'Behavior'}")
print("-" * 60)
for b in betas:
    loss_val, logit_val, _, _ = dpo_loss(lw, ll, rw, rl, b)
    prob = sigmoid(logit_val)
    kl = b * abs((lw - rw) + (ll - rl))  # rough KL scale
    if b < 0.05:
        behavior = "reward hack risk"
    elif b > 1.0:
        behavior = "too conservative"
    else:
        behavior = "good range"
    print(f"{b:<8} {loss_val:<10.4f} {prob:<12.4f} {kl:<14.4f} {behavior}")

print("\nKey: small β → aggressive optimization; large β → stay close to reference")

# --- 5d: DPO Training Loop ---
print("\n\n=== DPO Training Simulation ===\n")

# Simulate: 2D parameter space, learn to prefer y_w over y_l
np.random.seed(42)
theta = np.array([0.0, 0.0])  # policy parameters
theta_ref = theta.copy()       # frozen reference

# Preference data: 50 pairs with features
n_data = 50
x_feats = np.random.randn(n_data, 2)
# Preferred responses have positive feature 0, rejected have negative
y_w_feats = x_feats + np.array([1.0, 0.5])
y_l_feats = x_feats + np.array([-0.5, 0.3])

beta_train = 0.1
lr = 0.05
losses_dpo = []

for epoch in range(100):
    epoch_loss = 0
    grad = np.zeros(2)
    for i in range(n_data):
        # Log-probabilities (linear model: log π ∝ θ·y)
        log_pi_w_i = theta @ y_w_feats[i]
        log_pi_l_i = theta @ y_l_feats[i]
        log_ref_w_i = theta_ref @ y_w_feats[i]
        log_ref_l_i = theta_ref @ y_l_feats[i]
        
        ratio_diff = (log_pi_w_i - log_ref_w_i) - (log_pi_l_i - log_ref_l_i)
        logit_val = beta_train * ratio_diff
        
        loss_i = -np.log(sigmoid(logit_val) + 1e-10)
        epoch_loss += loss_i
        
        # Gradient
        weight = sigmoid(-logit_val)  # = 1 - σ(logit)
        grad += -beta_train * weight * (y_w_feats[i] - y_l_feats[i])
    
    theta -= lr * grad / n_data
    losses_dpo.append(epoch_loss / n_data)

print(f"{'Epoch':<8} {'Loss':<12} {'θ':<25} {'Prefers y_w?'}")
print("-" * 55)
for e in [0, 10, 30, 50, 99]:
    theta_snap = theta_ref + (theta - theta_ref) * (e+1) / 100  # interpolate
    pref_w = np.mean([theta @ y_w_feats[i] > theta @ y_l_feats[i] for i in range(n_data)])
    print(f"{e:<8} {losses_dpo[e]:<12.4f} [{theta[0]:.3f}, {theta[1]:.3f}]{'':<10} {pref_w*100:.0f}%")

# --- 5e: DPO vs PPO Comparison ---
print("\n\n=== DPO vs PPO Comparison ===\n")
print_table(
    ["Aspect", "PPO (RLHF)", "DPO"],
    [
        ["Models needed", "4 (policy, ref, reward, value)", "2 (policy, ref)"],
        ["Memory", "~36N bytes", "~8N bytes"],
        ["Hyperparameters", "Many (clip ε, GAE λ, KL coeff)", "Just β"],
        ["Training stability", "Difficult, reward hacking", "Stable, closed-form"],
        ["Online generation", "Required", "Not required"],
        ["Performance", "Strong when tuned well", "Competitive, simpler"],
    ]
)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(losses_dpo, 'b-', linewidth=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('DPO Loss')
    ax.set_title('DPO Training Loss')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('dpo_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: dpo_training.png")

## §6 GRPO — Group Relative Policy Optimisation

**Key idea (DeepSeek-R1):** No value model needed. For each prompt, sample a **group** of $G$ responses, normalise rewards within the group:

$$\hat{A}_i = \frac{r_i - \text{mean}(\{r_j\}_{j=1}^G)}{\text{std}(\{r_j\}_{j=1}^G)}$$

**GRPO Objective:**

$$\mathcal{L}_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^G \min\!\left(\frac{\pi_\theta(y_i|x)}{\pi_{\text{old}}(y_i|x)}\hat{A}_i, \; \text{clip}\!\left(\frac{\pi_\theta}{\pi_{\text{old}}}, 1\pm\epsilon\right)\hat{A}_i\right) + \beta \, D_{KL}(\pi_\theta \| \pi_{\text{ref}})$$

- **No critic/value function** → simpler, cheaper than PPO
- **Self-normalising** → no reward model scale issues

In [ ]:
# === §6: GRPO Implementation ===
np.random.seed(42)

# --- 6a: Group Advantage Normalisation ---
print("=== GRPO: Group Advantage Normalisation ===\n")

G = 8  # group size
n_prompts = 5

print(f"Group size G = {G}\n")
print(f"{'Prompt':<8} {'Raw rewards':<45} {'Normalised advantages'}")
print("-" * 95)

for p in range(n_prompts):
    # Sample G responses with different rewards
    rewards = np.random.uniform(-1, 3, G)
    
    # GRPO advantage: normalize within group
    mean_r = np.mean(rewards)
    std_r = np.std(rewards) + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    rew_str = "[" + ", ".join(f"{r:.2f}" for r in rewards) + "]"
    adv_str = "[" + ", ".join(f"{a:+.2f}" for a in advantages) + "]"
    print(f"{p+1:<8} {rew_str:<45} {adv_str}")

print("\n→ Zero mean within each group — no reward model scale issues!")

# --- 6b: GRPO Policy Update ---
print("\n\n=== GRPO Policy Update (Clipped) ===\n")

# Simulate policy ratio and clipping
epsilon = 0.2  # clip range
print(f"Clip range: [1-ε, 1+ε] = [{1-epsilon}, {1+epsilon}]\n")

# For one prompt, G=8 responses
rewards = np.array([3.0, 2.5, 1.0, 0.5, 0.0, -0.5, 1.5, 2.0])
mean_r, std_r = np.mean(rewards), np.std(rewards)
advantages = (rewards - mean_r) / (std_r + 1e-8)

# Simulated policy ratios (π_θ / π_old)
ratios = np.array([1.3, 1.1, 0.9, 0.7, 1.0, 1.2, 0.95, 1.05])

print(f"{'Response':<10} {'Reward':<10} {'Adv':<10} {'Ratio':<10} {'Clipped':<10} {'Obj':<10} {'Min(obj)'}")
print("-" * 70)
for i in range(G):
    obj_unclipped = ratios[i] * advantages[i]
    clipped_ratio = np.clip(ratios[i], 1 - epsilon, 1 + epsilon)
    obj_clipped = clipped_ratio * advantages[i]
    obj_final = min(obj_unclipped, obj_clipped) if advantages[i] >= 0 else max(obj_unclipped, obj_clipped)
    print(f"{i+1:<10} {rewards[i]:<10.1f} {advantages[i]:<+10.2f} {ratios[i]:<10.2f} {clipped_ratio:<10.2f} {obj_unclipped:<+10.3f} {obj_final:<+.3f}")

# --- 6c: GRPO Training Loop ---
print("\n\n=== GRPO Training Simulation ===\n")

# Simple task: learn to output high-reward actions
d_param = 4
theta = np.zeros(d_param)
theta_ref = theta.copy()
beta_kl = 0.01
epsilon_clip = 0.2
lr = 0.05

reward_history = []

for step in range(200):
    # "Generate" G responses per prompt (features)
    features = np.random.randn(G, d_param)
    
    # Log probs under current and old policy (linear model)
    log_pi = features @ theta
    log_pi_old = features @ theta.copy()  # π_old = π_θ at start of each step
    log_pi_ref = features @ theta_ref
    
    # Rewards: higher for responses with positive features[0] 
    rewards = features[:, 0] + 0.5 * features[:, 1] + np.random.randn(G) * 0.1
    
    # GRPO advantage
    adv = (rewards - np.mean(rewards)) / (np.std(rewards) + 1e-8)
    
    # Policy gradient with clipping
    ratios = np.exp(log_pi - log_pi_old)
    clipped = np.clip(ratios, 1 - epsilon_clip, 1 + epsilon_clip)
    
    surr1 = ratios * adv
    surr2 = clipped * adv
    obj = np.minimum(surr1, surr2)
    
    # KL penalty
    kl = np.mean(log_pi - log_pi_ref)
    
    # Gradient (simplified REINFORCE-style)
    grad = np.zeros(d_param)
    for i in range(G):
        weight = adv[i] * min(ratios[i], clipped[i])
        grad += weight * features[i]
    grad = -grad / G + beta_kl * 2 * (theta - theta_ref)
    
    theta -= lr * grad
    reward_history.append(np.mean(rewards))

print(f"{'Step':<8} {'Mean reward':<14} {'θ[0]':<10} {'θ[1]':<10} {'KL from ref'}")
print("-" * 55)
for s in [0, 20, 50, 100, 199]:
    kl_val = 0.5 * np.sum((theta - theta_ref)**2)  # approx
    print(f"{s:<8} {reward_history[s]:<14.3f} {theta[0]:<10.3f} {theta[1]:<10.3f} {kl_val:.4f}")

# --- 6d: GRPO Reward Design ---
print("\n\n=== GRPO Reward Design (DeepSeek-R1 Style) ===\n")
print_table(
    ["Reward Type", "Formula", "Use Case"],
    [
        ["Accuracy", "r = 1(answer correct)", "Math, code"],
        ["Format", "r = 1(follows format)", "Structured output"],
        ["Length penalty", "r = -max(0, len-limit)/limit", "Conciseness"],
        ["Composite", "r = 0.7·acc + 0.2·format + 0.1·length", "Production"],
        ["Process reward", "r = Σ step_correct / n_steps", "Chain-of-thought"],
    ]
)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(reward_history, 'b-', alpha=0.3, linewidth=0.5)
    # Rolling average
    window = 10
    rolling = np.convolve(reward_history, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(reward_history)), rolling, 'r-', linewidth=2, label=f'{window}-step avg')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Mean reward')
    axes[0].set_title('GRPO Training: Reward'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    # Advantage distribution
    rewards_demo = np.random.uniform(-1, 3, 1000)
    advs = (rewards_demo - np.mean(rewards_demo)) / (np.std(rewards_demo) + 1e-8)
    axes[1].hist(advs, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--', label='mean=0')
    axes[1].set_xlabel('Advantage'); axes[1].set_ylabel('Count')
    axes[1].set_title('GRPO: Normalised Advantage Distribution'); axes[1].legend()
    plt.tight_layout()
    plt.savefig('grpo_training.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: grpo_training.png")

## §7 Instruction Tuning & SFT

**SFT loss** — cross-entropy only on the **response** tokens (instruction is masked):

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|y|}\sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid x, y_{<t})$$

**IFD Score** — Instruction Following Difficulty (data quality metric):

$$\text{IFD}(x, y) = \frac{\mathcal{L}(y \mid x)}{\mathcal{L}(y)}$$

- IFD < 1: instruction **helps** → good training example
- IFD > 1: instruction **hurts** → likely noisy/contradictory pair

In [ ]:
# === §7: Instruction Tuning & SFT ===
np.random.seed(42)

# --- 7a: SFT Loss Masking Demo ---
print("=== SFT: Response-Only Loss Masking ===\n")

# Simulate a tokenised conversation
tokens = ["<s>", "[INST]", "What", "is", "2+2", "?", "[/INST]", "The", "answer", "is", "4", ".", "</s>"]
instruction_end = 6  # everything up to [/INST] is masked
V = 100  # vocab size

# Random logits (simulating model output)
logits = np.random.randn(len(tokens), V)
labels = np.random.randint(0, V, len(tokens))  # true next-token IDs

print(f"{'Pos':<5} {'Token':<12} {'Masked?':<10} {'Loss':<12} {'In gradient?'}")
print("-" * 50)

total_loss = 0
n_response = 0
for i in range(len(tokens)):
    # Cross-entropy for this position
    probs = softmax(logits[i])
    ce = -np.log(probs[labels[i]] + 1e-10)
    
    is_masked = i <= instruction_end
    in_grad = "No (instruction)" if is_masked else "Yes (response)"
    
    if not is_masked:
        total_loss += ce
        n_response += 1
    
    print(f"{i:<5} {tokens[i]:<12} {'✗ mask' if is_masked else '✓ keep':<10} {ce:<12.4f} {in_grad}")

print(f"\nSFT Loss = {total_loss/n_response:.4f} (avg over {n_response} response tokens)")
print(f"Masked {instruction_end+1} instruction tokens — their errors don't affect training!")

# --- 7b: IFD Score ---
print("\n\n=== IFD Score: Data Quality Filter ===\n")

# Simulate different instruction-response pairs
pairs = [
    ("Explain gravity",         "Gravity is the force...",  2.1, 3.5),  # good: instruction helps
    ("Translate to French",     "Bonjour le monde",         1.8, 2.0),  # marginal
    ("asdfgh random",            "The cat sat on mat",       3.2, 2.8),  # bad: instr hurts
    ("Write Python sort",       "def sort(lst):...",         1.5, 4.2),  # excellent
    ("Summarise: [10k words]",  "Summary here",             2.9, 2.5),  # bad
    ("What is 2+2?",            "4",                         0.3, 0.5),  # good, easy
]

print(f"{'Instruction':<25} {'L(y|x)':<10} {'L(y)':<10} {'IFD':<10} {'Quality'}")
print("-" * 65)
for instr, resp, loss_cond, loss_uncond in pairs:
    ifd = loss_cond / loss_uncond
    quality = "✓ Good" if ifd < 0.8 else ("~ Marginal" if ifd < 1.0 else "✗ Noisy")
    print(f"{instr:<25} {loss_cond:<10.2f} {loss_uncond:<10.2f} {ifd:<10.3f} {quality}")

print(f"\nFilter rule: keep pairs where IFD < 1.0 (instruction genuinely helps)")
print(f"→ Removes {sum(1 for _,_,lc,lu in pairs if lc/lu >= 1.0)}/{len(pairs)} low-quality pairs")

# --- 7c: Data Scaling ---
print("\n\n=== Instruction Tuning Data Scaling ===\n")
# Based on LIMA / Alpaca findings
data_sizes = [100, 500, 1000, 5000, 10000, 50000, 100000, 1000000]
# Simulate diminishing returns (log-linear improvement)
base_perf = 30  # base performance (no SFT)
scale_factor = 8  # log scaling

print(f"{'N samples':<12} {'Approx quality':<18} {'Marginal gain':<15} {'Note'}")
print("-" * 65)
prev_perf = base_perf
for n in data_sizes:
    perf = base_perf + scale_factor * np.log10(n) + np.random.randn() * 0.5
    perf = min(perf, 85)  # cap
    gain = perf - prev_perf
    note = ""
    if n == 1000:
        note = "LIMA sweet spot"
    elif n == 52000:
        note = "Alpaca size"
    elif n >= 100000:
        note = "diminishing returns"
    print(f"{n:<12,} {perf:<18.1f} {gain:<+15.1f} {note}")
    prev_perf = perf

print("\n→ Key insight (LIMA): 1,000 high-quality examples ≈ 50,000 noisy ones")

## §8 Continual Learning & Catastrophic Forgetting

**Elastic Weight Consolidation (EWC):**

$$\mathcal{L}_{\text{EWC}} = \mathcal{L}_{\text{new}}(\theta) + \frac{\lambda}{2}\sum_i F_i(\theta_i - \theta^*_i)^2$$

where:
- $F_i$ = diagonal of Fisher information matrix (importance of parameter $i$)
- $\theta^*$ = optimal parameters for previous task
- $\lambda$ = regularisation strength

**Fisher information:** $F_i = \mathbb{E}\left[\left(\frac{\partial \log p(y|x;\theta)}{\partial \theta_i}\right)^2\right]$

High $F_i$ → parameter $i$ was crucial for old task → penalise changes more

In [ ]:
# === §8: EWC & Continual Learning ===
np.random.seed(42)

# --- 8a: Catastrophic Forgetting Demo ---
print("=== Catastrophic Forgetting Demo ===\n")

# Task A: learn θ* ≈ [1.0, 2.0]
# Task B: learn θ* ≈ [3.0, -1.0]
# Without EWC: Task B overwrites Task A knowledge

theta = np.array([0.0, 0.0])
theta_star_A = np.array([1.0, 2.0])
theta_star_B = np.array([3.0, -1.0])
lr = 0.1

# Train on Task A
print("Training on Task A (target: [1.0, 2.0]):")
for step in range(50):
    grad = 2 * (theta - theta_star_A) + np.random.randn(2) * 0.05
    theta -= lr * grad
print(f"  After Task A: θ = [{theta[0]:.3f}, {theta[1]:.3f}]")
theta_after_A = theta.copy()

# Train on Task B (naive — catastrophic forgetting!)
print(f"\nTraining on Task B (target: [3.0, -1.0]) — NO protection:")
for step in range(50):
    grad = 2 * (theta - theta_star_B) + np.random.randn(2) * 0.05
    theta -= lr * grad
print(f"  After Task B: θ = [{theta[0]:.3f}, {theta[1]:.3f}]")
print(f"  Task A error:  {np.linalg.norm(theta - theta_star_A):.3f}  ← FORGOTTEN!")
print(f"  Task B error:  {np.linalg.norm(theta - theta_star_B):.3f}  ← Learned")

# --- 8b: EWC Implementation ---
print("\n\n=== EWC: Protecting Old Knowledge ===\n")

# Reset and re-do with EWC
theta = np.array([0.0, 0.0])

# Train Task A
for step in range(50):
    grad = 2 * (theta - theta_star_A) + np.random.randn(2) * 0.05
    theta -= lr * grad
theta_star = theta.copy()  # optimal for Task A
print(f"After Task A: θ* = [{theta_star[0]:.3f}, {theta_star[1]:.3f}]")

# Compute Fisher information (importance of each parameter)
# In practice: average squared gradients over Task A data
n_samples = 100
fisher = np.zeros(2)
for _ in range(n_samples):
    x_sample = np.random.randn(2)
    grad_log_p = 2 * (theta_star - theta_star_A) + x_sample * 0.1
    fisher += grad_log_p**2
fisher /= n_samples

print(f"Fisher information: F = [{fisher[0]:.4f}, {fisher[1]:.4f}]")
print(f"→ Parameter 0 importance: {fisher[0]:.4f}")
print(f"→ Parameter 1 importance: {fisher[1]:.4f}")

# Train Task B WITH EWC
lambda_ewc = 5.0
print(f"\nTraining Task B with EWC (λ = {lambda_ewc}):")

theta_ewc = theta_star.copy()
ewc_history = []

for step in range(50):
    # Task B loss gradient
    grad_task_b = 2 * (theta_ewc - theta_star_B) + np.random.randn(2) * 0.05
    
    # EWC penalty gradient: λ * F * (θ - θ*)
    grad_ewc = lambda_ewc * fisher * (theta_ewc - theta_star)
    
    # Total gradient
    grad_total = grad_task_b + grad_ewc
    theta_ewc -= lr * grad_total
    ewc_history.append(theta_ewc.copy())

print(f"  After Task B (EWC): θ = [{theta_ewc[0]:.3f}, {theta_ewc[1]:.3f}]")
print(f"  Task A error:  {np.linalg.norm(theta_ewc - theta_star_A):.3f}  ← PRESERVED!")
print(f"  Task B error:  {np.linalg.norm(theta_ewc - theta_star_B):.3f}")

# --- 8c: EWC Loss Calculation ---
print("\n\n=== EWC Loss: Worked Example ===\n")

theta_test = np.array([1.5, 0.5])
theta_star_test = np.array([1.0, 2.0])
F_test = np.array([10.0, 1.0])
lambda_test = 1.0

print(f"θ = {theta_test}")
print(f"θ* = {theta_star_test}")
print(f"F = {F_test}")
print(f"λ = {lambda_test}\n")

new_task_loss = 0.5  # pretend
ewc_penalty = 0.5 * lambda_test * np.sum(F_test * (theta_test - theta_star_test)**2)
total_loss = new_task_loss + ewc_penalty

print(f"L_new = {new_task_loss}")
for i in range(len(theta_test)):
    term = 0.5 * lambda_test * F_test[i] * (theta_test[i] - theta_star_test[i])**2
    print(f"EWC term {i}: 0.5 × {lambda_test} × {F_test[i]} × ({theta_test[i]}-{theta_star_test[i]})² = {term:.3f}")
print(f"EWC penalty = {ewc_penalty:.3f}")
print(f"Total loss = {new_task_loss} + {ewc_penalty:.3f} = {total_loss:.3f}")

# --- 8d: Comparison table ---
print("\n\n=== Continual Learning Methods ===\n")
print_table(
    ["Method", "Mechanism", "Memory Cost", "Performance"],
    [
        ["Naive FT", "No protection", "0", "Catastrophic forgetting"],
        ["L2 Regularisation", "‖θ-θ*‖²", "Store θ*", "Uniform penalty (suboptimal)"],
        ["EWC", "F·‖θ-θ*‖²", "Store θ*, F", "Parameter-aware protection"],
        ["Replay", "Mix old+new data", "Store old data", "Strong but expensive"],
        ["O-LoRA", "Orthogonal LoRA spaces", "Store LoRA bases", "No interference"],
    ]
)

# --- 8e: L2 vs EWC comparison ---
print("\n=== L2 vs EWC ===\n")
theta_l2 = theta_star.copy()
lambda_l2 = 5.0

for step in range(50):
    grad_b = 2 * (theta_l2 - theta_star_B) + np.random.randn(2) * 0.05
    grad_l2_reg = lambda_l2 * (theta_l2 - theta_star)  # uniform penalty
    theta_l2 -= lr * (grad_b + grad_l2_reg)

print(f"{'Method':<12} {'θ final':<25} {'Task A err':<14} {'Task B err'}")
print("-" * 60)
methods = [
    ("Naive", np.array([theta_star_B[0]+0.01, theta_star_B[1]-0.02])),
    ("L2-reg", theta_l2),
    ("EWC", theta_ewc),
]
for name, t in methods:
    err_a = np.linalg.norm(t - theta_star_A)
    err_b = np.linalg.norm(t - theta_star_B)
    print(f"{name:<12} [{t[0]:.3f}, {t[1]:.3f}]{'':<12} {err_a:<14.3f} {err_b:.3f}")

print("\n→ EWC protects important params (high F) while allowing unimportant ones to change")

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    # Trajectory plot
    ewc_arr = np.array(ewc_history)
    axes[0].plot(*theta_star_A, 'g*', markersize=15, label='Task A optimal')
    axes[0].plot(*theta_star_B, 'r*', markersize=15, label='Task B optimal')
    axes[0].plot(ewc_arr[:, 0], ewc_arr[:, 1], 'b-o', markersize=3, alpha=0.5, label='EWC path')
    axes[0].plot(*theta_ewc, 'bs', markersize=10, label='EWC final')
    axes[0].set_xlabel('θ₀'); axes[0].set_ylabel('θ₁')
    axes[0].set_title('EWC Training Trajectory'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    # Fisher importance
    axes[1].bar(['θ₀', 'θ₁'], fisher, color=['steelblue', 'coral'])
    axes[1].set_ylabel('Fisher Information')
    axes[1].set_title('Parameter Importance (Fisher)')
    axes[1].grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('ewc_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: ewc_analysis.png")

## §9 Model Merging

Combine capabilities of multiple fine-tuned models **without retraining**:

**Task vectors:** $\tau_k = \theta_k - \theta_0$ (what each fine-tune learned)

**Methods:**
| Method | Formula | Key Idea |
|--------|---------|----------|
| **Weight Average** | $\theta = \frac{1}{K}\sum_k\theta_k$ | Simple mean |
| **SLERP** | Spherical interpolation | Preserves norm |
| **TIES** | Trim+Elect+Merge signs | Resolves sign conflicts |
| **DARE** | Random drop + rescale | Sparse merging |

**SLERP:**
$$\theta_{\text{merge}} = \frac{\sin((1-t)\Omega)}{\sin\Omega}\theta_1 + \frac{\sin(t\Omega)}{\sin\Omega}\theta_2, \quad \Omega = \arccos\!\left(\frac{\theta_1 \cdot \theta_2}{\|\theta_1\|\|\theta_2\|}\right)$$

In [ ]:
# === §9: Model Merging ===
np.random.seed(42)

# --- 9a: Task Vectors ---
print("=== Task Vectors ===\n")
d = 8  # weight dimension

# Base model
theta_0 = np.random.randn(d) * 0.5

# Fine-tuned models (different tasks)
theta_math = theta_0 + np.array([0.3, -0.1, 0.5, 0.0, 0.2, -0.3, 0.1, 0.4])
theta_code = theta_0 + np.array([-0.1, 0.4, 0.0, 0.6, -0.2, 0.1, 0.3, -0.1])
theta_chat = theta_0 + np.array([0.1, 0.2, -0.1, 0.1, 0.3, 0.2, -0.2, 0.1])

# Task vectors
tau_math = theta_math - theta_0
tau_code = theta_code - theta_0
tau_chat = theta_chat - theta_0

print("Task vectors (τ = θ_ft - θ_0):")
print(f"  τ_math: {print_vector(tau_math)}")
print(f"  τ_code: {print_vector(tau_code)}")
print(f"  τ_chat: {print_vector(tau_chat)}")

# Cosine similarity between task vectors
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nTask vector similarity:")
print(f"  math-code: {cosine_sim(tau_math, tau_code):.3f}")
print(f"  math-chat: {cosine_sim(tau_math, tau_chat):.3f}")
print(f"  code-chat: {cosine_sim(tau_code, tau_chat):.3f}")

# --- 9b: Weight Averaging ---
print("\n\n=== Method 1: Weight Averaging ===\n")
theta_avg = (theta_math + theta_code + theta_chat) / 3
tau_avg = theta_avg - theta_0

print(f"θ_merged = mean(θ_math, θ_code, θ_chat)")
print(f"τ_merged: {print_vector(tau_avg)}")
print(f"‖τ_merged‖ = {np.linalg.norm(tau_avg):.4f}")
print(f"‖τ_math‖   = {np.linalg.norm(tau_math):.4f}")
print(f"→ Averaging shrinks task vectors (destructive interference)")

# --- 9c: SLERP ---
print("\n\n=== Method 2: SLERP (Spherical Linear Interpolation) ===\n")

def slerp(theta1, theta2, t):
    """Spherical interpolation between two weight vectors."""
    cos_omega = np.dot(theta1, theta2) / (np.linalg.norm(theta1) * np.linalg.norm(theta2))
    cos_omega = np.clip(cos_omega, -1, 1)
    omega = np.arccos(cos_omega)
    
    if abs(np.sin(omega)) < 1e-10:
        return (1 - t) * theta1 + t * theta2  # fallback to LERP
    
    w1 = np.sin((1 - t) * omega) / np.sin(omega)
    w2 = np.sin(t * omega) / np.sin(omega)
    return w1 * theta1 + w2 * theta2

print(f"{'t':<8} {'‖θ_slerp‖':<14} {'‖θ_lerp‖':<14} {'SLERP preserves norm?'}")
print("-" * 55)
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    theta_s = slerp(theta_math, theta_code, t)
    theta_l = (1 - t) * theta_math + t * theta_code  # linear interpolation
    norm_s = np.linalg.norm(theta_s)
    norm_l = np.linalg.norm(theta_l)
    preserves = "✓" if abs(norm_s - np.linalg.norm(theta_math)) < 0.1 else "~"
    print(f"{t:<8} {norm_s:<14.4f} {norm_l:<14.4f} {preserves}")

print(f"\n‖θ_math‖ = {np.linalg.norm(theta_math):.4f}")
print(f"‖θ_code‖ = {np.linalg.norm(theta_code):.4f}")
print("→ SLERP preserves magnitude; LERP shrinks at midpoint")

# --- 9d: TIES Merging ---
print("\n\n=== Method 3: TIES (Trim, Elect Sign, Merge) ===\n")

def ties_merge(task_vectors, density=0.5):
    """TIES merging: trim small values, resolve sign conflicts, merge."""
    K = len(task_vectors)
    d = len(task_vectors[0])
    
    # Step 1: TRIM — keep only top-k% by magnitude
    trimmed = []
    for tv in task_vectors:
        threshold = np.percentile(np.abs(tv), (1 - density) * 100)
        tv_trimmed = np.where(np.abs(tv) >= threshold, tv, 0)
        trimmed.append(tv_trimmed)
    
    # Step 2: ELECT SIGN — majority vote for each parameter
    signs = np.zeros(d)
    for j in range(d):
        pos = sum(1 for tv in trimmed if tv[j] > 0)
        neg = sum(1 for tv in trimmed if tv[j] < 0)
        signs[j] = 1 if pos >= neg else -1
    
    # Step 3: MERGE — average values with elected sign
    merged = np.zeros(d)
    for j in range(d):
        vals = [tv[j] for tv in trimmed if np.sign(tv[j]) == signs[j] and tv[j] != 0]
        if vals:
            merged[j] = np.mean(vals)
    
    return merged

tau_ties = ties_merge([tau_math, tau_code, tau_chat], density=0.7)
theta_ties = theta_0 + tau_ties

print(f"Trimmed + sign-elected + merged task vector:")
print(f"  τ_TIES: {print_vector(tau_ties)}")
print(f"  ‖τ_TIES‖ = {np.linalg.norm(tau_ties):.4f}  (vs avg ‖τ‖ = {np.linalg.norm(tau_avg):.4f})")

# --- 9e: Comparison ---
print("\n\n=== Merging Methods Comparison ===\n")
print_table(
    ["Method", "‖τ_merged‖", "Preserves norm?", "Sign conflicts?", "Sparsity-aware?"],
    [
        ["Averaging", f"{np.linalg.norm(tau_avg):.3f}", "No (shrinks)", "No", "No"],
        ["SLERP", f"{np.linalg.norm(slerp(theta_math,theta_code,0.5)-theta_0):.3f}", "Yes", "No", "No"],
        ["TIES", f"{np.linalg.norm(tau_ties):.3f}", "Partially", "Yes", "Yes"],
    ]
)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Task vectors heatmap
    tv_matrix = np.array([tau_math, tau_code, tau_chat, tau_avg, tau_ties])
    im = axes[0].imshow(tv_matrix, cmap='RdBu', aspect='auto', vmin=-0.6, vmax=0.6)
    axes[0].set_yticks(range(5))
    axes[0].set_yticklabels(['τ_math', 'τ_code', 'τ_chat', 'τ_avg', 'τ_TIES'])
    axes[0].set_xlabel('Parameter index'); axes[0].set_title('Task Vectors')
    plt.colorbar(im, ax=axes[0])
    # SLERP vs LERP norms
    ts = np.linspace(0, 1, 50)
    norms_s = [np.linalg.norm(slerp(theta_math, theta_code, t)) for t in ts]
    norms_l = [np.linalg.norm((1-t)*theta_math + t*theta_code) for t in ts]
    axes[1].plot(ts, norms_s, 'r-', linewidth=2, label='SLERP')
    axes[1].plot(ts, norms_l, 'b--', linewidth=2, label='LERP')
    axes[1].set_xlabel('Interpolation t'); axes[1].set_ylabel('‖θ‖')
    axes[1].set_title('SLERP vs LERP: Norm Preservation')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('model_merging.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: model_merging.png")

## §10 Summary & Method Selection Guide

The fine-tuning landscape in 2024-2025 is rich with methods. Here's a practical decision framework:

In [ ]:
# === §10: Summary ===
print("=" * 80)
print("Fine-Tuning Mathematics — Complete Method Summary")
print("=" * 80)

print("\n--- Method Selection Guide ---\n")
print_table(
    ["Method", "Params Trained", "Memory", "Best For", "Key Equation"],
    [
        ["Full FT", "100%", "16N", "Max quality, large data", "SGD on all θ"],
        ["LoRA", "0.01-1%", "~2N + adapters", "Standard PEFT", "W' = W₀ + BA"],
        ["QLoRA", "0.01-1%", "~0.5N + adapters", "GPU-constrained", "NF4 + LoRA"],
        ["DoRA", "0.01-1% + d", "~2N + adapters", "Better than LoRA", "m·V/‖V‖"],
        ["DPO", "All (usually LoRA)", "~8N", "Preference alignment", "-log σ(β·Δr)"],
        ["GRPO", "All (usually LoRA)", "~4N", "Reasoning, math", "Â = (r-μ)/σ"],
        ["SFT", "All (usually LoRA)", "~8N", "Instruction following", "CE on response"],
        ["EWC", "All", "16N + F", "Continual learning", "L + λ·F·(θ-θ*)²"],
    ]
)

print("\n--- Decision Flowchart ---\n")
print("""
┌─ Do you have preference data?
│  ├─ Yes → Have verifiable rewards? → Yes → GRPO
│  │                                  → No  → DPO
│  └─ No  → Have instruction data?
│           ├─ Yes → SFT (then optionally DPO/GRPO)
│           └─ No  → Continue pretraining
│
├─ GPU memory constraint?
│  ├─ 1 GPU (24GB) → QLoRA, r=16-32
│  ├─ 1 GPU (80GB) → LoRA, r=32-64 on ≤13B
│  └─ Multi-GPU    → Full FT or LoRA on large models
│
└─ Multiple tasks?
   ├─ Sequential → EWC or O-LoRA
   └─ Combine    → Model Merging (TIES/DARE)
""")

print("\n--- Key Takeaways ---\n")
takeaways = [
    "1. LoRA works because fine-tuning updates have LOW INTRINSIC RANK",
    "2. QLoRA enables fine-tuning 70B models on a single GPU",
    "3. DPO eliminates the reward model — just needs preference pairs",
    "4. GRPO eliminates the value model — just needs reward function",
    "5. Fisher Information identifies which parameters matter most",
    "6. Model merging combines skills WITHOUT retraining",
    "7. β in DPO controls exploration vs. staying near reference",
    "8. Scaling factor α/r in LoRA controls adaptation speed",
    "9. IFD score filters low-quality instruction data",
    "10. The full pipeline: Pretrain → SFT → DPO/GRPO"
]
for t in takeaways:
    print(f"  {t}")

print("\n" + "=" * 80)
print("End of Fine-Tuning Mathematics Theory Notebook")
print("=" * 80)